In [2]:
import numpy as np
import pandas as pd
import time
from scipy.sparse import csr_matrix
import seaborn as sns
import hypernetx as hnx
import matplotlib.pyplot as plt
from pathlib import Path
import numpy as np
import pandas as pd
import time
from dataclasses import dataclass
import json


pd.set_option("display.max_rows", 999)

In [ ]:

@dataclass
class HyperNode:
    id: int
    name: str
    type: int


@dataclass
class HyperEdge:
    nodes: list[int]
    weight: float


@dataclass
class HyperGraph:
    edges: list[HyperEdge]
    nodes: list[HyperNode]
    metadata: dict

    def check(self):
        assert self.metadata['num_nodes'] == self.num_nodes, \
            f"Metadata num_nodes {self.metadata['num_nodes']} != actual num_nodes {self.num_nodes}"
        assert self.metadata['num_edges'] == self.num_edges, \
            f"Metadata num_edges {self.metadata['num_edges']} != actual num_edges {self.num_edges}"
        print('Hypergraph OK')

    @property
    def num_nodes(self):
        return len(self.nodes)

    @property
    def num_edges(self):
        return len(self.edges)

    def __str__(self):
        return f'HyperGraph(num_nodes={self.num_nodes}, num_edges={self.num_edges})'

    def __repr__(self):
        return str(self.metadata)


def load_graph(dir_path: Path):
    assert dir_path.exists(), f'Path {dir_path} does not exist.'

    with open(dir_path / 'metadata.json', 'r') as f:
        metadata = json.load(f)
    begin = time.time()
    df_edges = pd.read_csv(dir_path / 'edges.csv').astype(int)
    print(f'Load edges in {time.time() - begin:.2f} seconds.')

    df_nodes = pd.read_csv(dir_path / 'nodes.csv', index_col=0)
    print('Load nodes and edges from', dir_path)
    print(f'Num nodes: {len(df_nodes)}, Num edges: {len(df_edges)}')
    print('Metadata:', metadata)
    nodes = [HyperNode(id=i, type=row.Type, name=row.Name) for i, row in df_nodes.iterrows()]
    edges = [HyperEdge(nodes=[row.Omics, row.Gene, row.Cell], weight=row.Weight)
             for _, row in df_edges.iterrows()]
    graph = HyperGraph(nodes=nodes, edges=edges, metadata=metadata)
    return graph


In [9]:
g = load_graph(Path('./hypergraph/PEA_STA'))
print(g)

Load edges in 0.02 seconds.
Load nodes and edges from hypergraph/PEA_STA
Num nodes: 378, Num edges: 34860
Metadata: {'num_cells': 210, 'num_genes': 166, 'num_omics': 2, 'num_nodes': 378, 'num_edges': 34860, 'omics_offset': 0, 'gene_offset': 2, 'cell_offset': 168, 'raw_data': {'expr': './data/PEA_STA/expression_data.csv', 'protein': './data/PEA_STA/protein_data.csv'}, 'cell_key': 'Cells '}
HyperGraph(num_nodes=378, num_edges=34860)


In [12]:
g.nodes[-1]

HyperNode(id=377, name='u3035_6d_contol_H12', type=2)

In [10]:
g.check()